<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/2_02_database_partitioning_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.02 Database Partitioning and Replication
### *What changes when data lives on several machines? A simulated cluster*

#### **This Notebook is very optional. The previous probably already took you the hour I am asking you to do.**

The first notebook ran everything on one machine, and may have given the impression that the non-relational model is just inferior. You had to do more work (schema on read) and no-one likes doing more work.

The main reason many NoSQL systems exist (beyond some of the difference we already saw in the last notebook) is different: **the data does not fit on one machine, or must stay available when a machine fails.** You cannot see that on a single node, so here we simulate a small cluster and watch what changes.

Note, this tutorial is intentionally more complicated than anything we have seen yet, and unapologetically so. If you are new to all of this (databases, code, etc.) and it doesn't make any sense ...

![](https://drive.google.com/uc?export=view&id=1q4uBhzWnXvuezwdgLu9sKGduxZJ433Kx)

... this is not something we will test you on or that strictly you will need in many walks of analytics life. If you are more familiar with this sort of stuff and/or more interested in working in areas that include database selection, then ... enjoy (I guess).

You also absolutely have permission to copy this notebook into an AI to ask questions about the code or ideas.

By the end you should be able to:
1. Explain how data is **partitioned** (sharded) across nodes and why the choice of partition key matters.
2. Explain why some queries touch one node and others must **fan out** to all of them, and how partial results are merged.
3. Show what happens when a node fails, with and without **replication**.
4. Explain a **stale read** and when it is acceptable.

## Read this first: what is real and what is modelled
- **Nodes.** Each "node" is an in-memory SQLite database inside this one Colab process. They are not separate machines.
- **Time per node is measured** for real (how long that node's query took).
- **The network is modelled.** We add a fixed `NET_MS` = 2 ms (milliseconds) for every round trip. That number is an assumption we chose, not a measurement.
- **Parallelism is modelled.** When a query goes to several nodes we run them one after another but *report* the cost as if they ran at the same time: `NET_MS + the slowest node + merge time`.
- **Ignored entirely:** leader election and consensus, network partitions, clocks, rebalancing, security, disk durability, concurrent users. Real systems (Cassandra, MongoDB and DynamoDB are well-known examples) spend most of their complexity on exactly these.

Treat the numbers as illustrations of a mechanism (and a general principle), not as evidence about any product.

## 1. Set up

In [ ]:
import time, zlib, sqlite3, statistics
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# CONSTANT VARIABLES
NET_MS = 2.0                 # modelled network round trip per request (an assumption)
N_ROWS = 100_000             # number of rows
rng = np.random.default_rng(42)
# seed. This just means the random numbers work the same way each time.

# data items - our data is a set of stations that have a number of visits
STATIONS = ["Market Square", "Railway Station", "Canal Basin", "University Gate", "Park Road",
            "Library Corner", "Hospital Entrance", "Riverside Walk", "Old Town Hall",
            "Retail Park", "Sports Centre", "Cathedral Close"]

# Deliberately uneven: a few stations are far busier than the rest, like real "hot" keys
# raw number of visits
station_p = np.array([30, 18, 10, 8, 7, 6, 5, 4, 4, 3, 3, 2], float)
# we represent each station probablity as the number above divided by the total of all visits.
# algebraically station_i_probability = station_i_visits / sum(station_visits)
station_p /= station_p.sum()

# make this into a dataframe
df = pd.DataFrame({
    "trip_id":      np.arange(1, N_ROWS + 1),     # ids autoincrement - 1, 2, 3, etc.
    "bike_id":      rng.integers(1, 501, N_ROWS), # as above
    "station":      rng.choice(STATIONS, N_ROWS, p=station_p), # randomly pick a station according to its probablity
    "user_type":    rng.choice(["Member", "Casual"], N_ROWS, p=[0.65, 0.35]), # 65% chance its a member, 35% not
    "start_hour":   rng.integers(0, 24, N_ROWS),  # random time
    "duration_min": rng.gamma(2.0, 8.0, N_ROWS).round(1), # random duration
})
KEYS = [int(k) for k in rng.integers(1, N_ROWS + 1, 40)]  # create an id/key for each value generated above

# print number of trips - should be 100k
print(f"{len(df):,} trips")

# print the number of visits of the first three stations
print((df["station"].value_counts(normalize=True).head(3) * 100).round(1).to_string())

## 2. Two small functions
**Partitioning.** To decide which node owns a row, we take the value of the **partition key**, turn it into a number with a hash function (hash partioning from the slides), and take the remainder after dividing by the number of nodes. I.e. if we have 4 nodes (n0 to n3) and hash function gives us the number 6, we allocate to machine n1. (If our hash gave us 1 that would be n0; 2 would be n1; 3 would be n2; 4 would be n3; 5 goes back to the start so n0; 6 gives us n1).

The same key then always goes to the same node, and keys spread roughly evenly (if the key values themselves are spread evenly).

`zlib.crc32(text.encode())` gives a stable number for a piece of text. (Python's built-in `hash()` changes between runs, which would be a bug in a database.)

In [ ]:
def hash_to_node(value, n_nodes):
    """Return which node (0 .. n_nodes-1) should own this partition-key value."""
    return zlib.crc32(str(value).encode()) % n_nodes

We'll test this works by create `a` and `b` based on the same data (the station "Park Road" and a value of 4).

If it has worked then `a` and `b` will have the same value (`a==b` is `True`). Also, `a` should have a value that is 0 or greater (first node - n0) and a value no greater than 3 (last node - n3) ... `0 <= a < 4`. Lastly, any value we draw between 0 and 1,000 should return either 0, 1, 2 or 3 (the nodes). I.e. `hash_to_node(i, 4) for i in range(1000)} == {0, 1, 2, 3}`

In [ ]:
# Quick checks (should print True True True)
a, b = hash_to_node("Park Road", 4), hash_to_node("Park Road", 4)
print(a == b, 0 <= a < 4, {hash_to_node(i, 4) for i in range(1000)} == {0, 1, 2, 3})

Three `True` values so everything looks OK.

**Merging partial results.** If each node returns, for each station, the **sum** of durations and the **count** of trips, the coordinator can combine them. Write `merge_partials` so it returns the average duration per station across all nodes.

Each partial is a list of `(station, sum_of_durations, count)` tuples; you are given a list of such lists.

Think carefully: would it be correct to average the averages the nodes calculate? The check below tests that.

In [ ]:
def merge_partials(partials):
    """partials: list of lists of (station, total_duration, trip_count). Return {station: average_duration}."""
    totals, counts = {}, {}
    for part in partials:
        for station, total, count in part:
            totals[station] = totals.get(station, 0) + total
            counts[station] = counts.get(station, 0) + count
    return {s: totals[s] / counts[s] for s in totals}

In [ ]:
# Node 1 saw one A trip totalling 10 min; node 2 saw four A trips totalling 20 min
test = merge_partials([[("A", 10, 1)], [("A", 20, 4), ("B", 5, 1)]])
print(test)                       # expect A = 6.0 and B = 5.0
print("average of the two node averages would give A =", (10 / 1 + 20 / 4) / 2, "(wrong)")

## 3. The cluster
Read this code rather than trying to write it. Its quite complicated, and I haven't excessively commented it because it isn't necessarily to fully understand everything here. These are more helper functions for the simulation than code you would necessarily write in the real-world.

The key ideas:
- A `Node` is a small database that can be switched off (`alive = False`).
- Every row is tagged with its **partition number** `part`, worked out from the partition key.
- With replication factor `rf = 2`, a row is also copied to the **next** node, so it lives on two nodes (so we have a backup).
- `get_trip` and `avg_by_station` show the two kinds of query: go to **one** node, or ask **all** of them and merge.
- Each call returns a `Result` with the answer, the **modelled cost in ms** and the **number of requests** sent.

In [ ]:
class NodeDown(Exception):
    pass

@dataclass
class Result:
    value: object
    ms: float                     # modelled cost: NET_MS + slowest node + merge
    requests: int
    missing: list = field(default_factory=list)   # partitions that could not be read

class Node:
    def __init__(self, name):
        self.name, self.alive = name, True
        self.con = sqlite3.connect(":memory:")
        self.con.execute("""CREATE TABLE trips (trip_id INTEGER PRIMARY KEY, bike_id INT, station TEXT,
                            user_type TEXT, start_hour INT, duration_min REAL, part INT)""")
        self.con.execute("CREATE INDEX idx_part ON trips(part)")

    def _timed(self, fn):
        if not self.alive:
            raise NodeDown(self.name)
        t0 = time.perf_counter()
        out = fn()
        return out, (time.perf_counter() - t0) * 1000

    def insert_many(self, rows):
        def go():
            # bulk insert
            self.con.executemany("INSERT OR REPLACE INTO trips VALUES (?,?,?,?,?,?,?)", rows)
            self.con.commit()
        return self._timed(go)

    def get(self, trip_id):
        return self._timed(lambda: self.con.execute("SELECT * FROM trips WHERE trip_id = ?", (trip_id,)).fetchone())

    def partial_by_station(self, part):
        return self._timed(lambda: self.con.execute(
            "SELECT station, SUM(duration_min), COUNT(*) FROM trips WHERE part = ? GROUP BY station", (part,)).fetchall())

    def count(self):
        return self._timed(lambda: self.con.execute("SELECT COUNT(*) FROM trips").fetchone()[0])[0]


class Cluster:
    def __init__(self, n_nodes, rf=1, partition_key="trip_id"):
        self.nodes = [Node(f"node{i}") for i in range(n_nodes)]
        self.rf = min(rf, n_nodes)
        self.partition_key = partition_key

    def part_of(self, key_value):
        return hash_to_node(key_value, len(self.nodes))

    def replica_nodes(self, part):
        """The nodes holding copies of this partition: the owner first, then the next rf-1 nodes."""
        return [(part + k) % len(self.nodes) for k in range(self.rf)]

    def load(self, data):
        data = data.copy()
        data["part"] = [self.part_of(v) for v in data[self.partition_key]]
        n = len(self.nodes)
        for k in range(self.rf):
            target = (data["part"] + k) % n
            for idx, chunk in data.groupby(target):
                self.nodes[idx].insert_many(list(chunk.itertuples(index=False, name=None)))

    def rows_per_node(self):
        return [node.count() for node in self.nodes]

    # ---- queries -------------------------------------------------------
    def get_trip(self, trip_id):
        if self.partition_key == "trip_id":                      # we know exactly where it lives
            for idx in self.replica_nodes(self.part_of(trip_id)):
                try:
                    row, ms = self.nodes[idx].get(trip_id)
                    return Result(row, NET_MS + ms, 1)
                except NodeDown:
                    continue                                     # fail over to the next copy
            return Result(None, NET_MS, 1, missing=[self.part_of(trip_id)])
        times, found = [], None                                  # key is not the partition key: ask every node
        for node in self.nodes:
            try:
                row, ms = node.get(trip_id)
                times.append(ms); found = found or row
            except NodeDown:
                pass
        return Result(found, NET_MS + max(times, default=0), len(times))

    def avg_by_station(self, allow_partial=False):
        per_node_ms, partials, missing = {}, [], []
        for part in range(len(self.nodes)):
            for idx in self.replica_nodes(part):
                try:
                    rows, ms = self.nodes[idx].partial_by_station(part)
                    partials.append(rows)
                    per_node_ms[idx] = per_node_ms.get(idx, 0) + ms   # a node serving 2 partitions does 2 jobs
                    break
                except NodeDown:
                    continue
            else:
                missing.append(part)
        if missing and not allow_partial:
            raise NodeDown(f"partitions {missing} are unavailable")
        t0 = time.perf_counter()
        value = merge_partials(partials)
        merge_ms = (time.perf_counter() - t0) * 1000
        return Result(value, NET_MS + max(per_node_ms.values(), default=0) + merge_ms, len(partials), missing)

    def insert_trip(self, row):
        """row: dict with the same fields as df. Writes to every replica."""
        part = self.part_of(row[self.partition_key])
        tup = (row["trip_id"], row["bike_id"], row["station"], row["user_type"],
               row["start_hour"], row["duration_min"], part)
        times = []
        for idx in self.replica_nodes(part):
            try:
                times.append(self.nodes[idx].insert_many([tup])[1])
            except NodeDown:
                pass
        return Result(None, NET_MS + max(times, default=0), len(times))

Honestly, don't panic if that didn't make perfect sense, you can see all this code in use below.

## 4. Build four clusters on the same data
| Name | Nodes | Partition key | Copies of each row |
| --- | --- | --- | --- |
| A | 1 | trip_id | 1 |
| B | 4 | trip_id | 1 |
| C | 4 | station | 1 |
| D | 4 | trip_id | 2 |

* A is a single-machine baseline (not distributed);
* B splits using hash partioning based on trip_id with no replication;
* C splits using hash partioning based on station with no replication;
* D splits using hash partioning based on trip_id and this time replicates every node's data to a different node.

In [ ]:
clusters = {
    "A: 1 node":                  Cluster(1),
    "B: 4 nodes, key=trip_id":    Cluster(4, partition_key="trip_id"),
    "C: 4 nodes, key=station":    Cluster(4, partition_key="station"),
    "D: 4 nodes, key=trip_id, 2 copies": Cluster(4, rf=2, partition_key="trip_id"),
}
for name, c in clusters.items():
    c.load(df)
    print(f"{name:36s} rows per node: {c.rows_per_node()}")

**Predict before you look at the chart:** which cluster has the most uneven spread of rows? Why might that matter for speed?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
for ax, name in zip(axes, ["B: 4 nodes, key=trip_id", "C: 4 nodes, key=station", "D: 4 nodes, key=trip_id, 2 copies"]):
    counts = clusters[name].rows_per_node()
    ax.bar([f"node{i}" for i in range(4)], counts, color="#0072B2", width=0.6)
    ax.set_title(name.split(",", 1)[0] + "," + name.split(",", 1)[1], fontsize=8, loc="left")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("rows stored")
plt.tight_layout(); plt.show()

* Approach A is just one machine so just 100k rows (not shown here).
* Approach B (partioning on ID using hash) means exactly 25k on each node - and even quarter.
* Apporach C (partioning on Station using) gives a different number of rows on each node. This is because the number of trips is different for each station.
* Approach D is the same as B but with replication. Hence there are exactly 50k on each node - two quarters on each.

## 5. Compare the clusters
For each cluster we measure, using the modelled cost:
- **Lookup** of one trip by `trip_id` (average of 40 different keys)
- **Aggregate**: average duration per station (median of 5 runs)
- **Insert** one new trip (average of 20)
- **Requests**: how many nodes had to be contacted

In [ ]:
def measure(cluster, base_id=10_000_000):
    look = [cluster.get_trip(k) for k in KEYS]
    agg = [cluster.avg_by_station() for _ in range(5)]
    ins = [cluster.insert_trip({"trip_id": base_id + i, "bike_id": 1, "station": "Park Road",
                                "user_type": "Member", "start_hour": 9, "duration_min": 10.0}) for i in range(20)]
    rows = cluster.rows_per_node()
    return {
        "lookup ms":    round(statistics.mean(r.ms for r in look), 2),
        "lookup requests": round(statistics.mean(r.requests for r in look), 1),
        "aggregate ms": round(statistics.median(r.ms for r in agg), 2),
        "insert ms":    round(statistics.mean(r.ms for r in ins), 2),
        "insert requests": round(statistics.mean(r.requests for r in ins), 1),
        "busiest node / average": round(max(rows) / (sum(rows) / len(rows)), 2),
        "total rows stored": sum(rows),
    }

summary = pd.DataFrame({name: measure(c) for name, c in clusters.items()}).T
summary

**Check the answers are right before you trust the speed.** All four clusters should give the same average per station (cluster D stores extra copies, which must not be double-counted, but does mean our insert requests double).

We will test if the result of this query (average trips by station) is the same for B to D as it is on A (the baseline single node):

In [ ]:
ref = clusters["A: 1 node"].avg_by_station().value
for name, c in clusters.items():
    v = c.avg_by_station().value
    ok = all(abs(v[s] - ref[s]) < 1e-6 for s in ref) and len(v) == len(ref)
    print(f"{name:36s} matches single node: {ok}")

Now compare the clusters in the table above. Note which numbers fell for the aggregate, which stayed flat for the lookup, and where cluster C is worse than B. Then write your explanations in Part 8.

**Look at `lookup requests` as well as `lookup ms`.** The modelled lookup time is almost the same everywhere because we assume nodes work at the same moment, but the number of requests is the load placed on the cluster.

**Reminder:** all of these costs include a modelled network delay and an assumption that nodes run at the same time. They show a mechanism, not what a real system would achieve.

## 6. A node fails
We switch off `node1` in two clusters and look at what the user would see: B has one copy of each row; D has two.

In [ ]:
def failure_report(name):
    c = clusters[name]
    c.nodes[1].alive = False
    failed = sum(c.get_trip(k).value is None for k in range(1, 401))
    true = clusters["A: 1 node"].avg_by_station().value
    try:
        c.avg_by_station()
        strict = "answered"
    except NodeDown as e:
        strict = f"refused ({e})"
    part = c.avg_by_station(allow_partial=True)
    drift = max(abs(part.value[s] - true[s]) for s in part.value)
    covered = 100 * sum(c.part_of(t) not in part.missing for t in df["trip_id"]) / len(df)
    print(f"{name}")
    print(f"  lookups that failed (400 tried):   {failed}")
    print(f"  aggregate, strict mode:            {strict}")
    print(f"  aggregate, allow_partial=True:     missing partitions {part.missing}, "
          f"largest error in any station average = {drift:.3f} min, cost {part.ms:.1f} ms")
    print(f"  share of all trips actually included in that answer: {covered:.1f}%")
    c.nodes[1].alive = True

failure_report("B: 4 nodes, key=trip_id")
failure_report("D: 4 nodes, key=trip_id, 2 copies")

Look carefully at B's `allow_partial=True` line. The query "worked" and returned numbers. What does it **not** tell the user? When would returning a partial answer be acceptable, and when dangerous? (You will answer in Part 8.)

## 7. Replication lag and stale reads
Copying data to a second node takes time. This tiny model has a primary and a replica; writes go to the primary immediately and reach the replica only when `sync()` runs.

In [ ]:
class LaggyPair:
    def __init__(self, data):
        self.primary, self.replica, self.pending = dict(data), dict(data), []
    def write(self, key, value):
        self.primary[key] = value
        self.pending.append((key, value))
    def sync(self):
        for k, v in self.pending:
            self.replica[k] = v
        self.pending.clear()

pair = LaggyPair({"balance": 100, "likes": 40})
pair.write("balance", 50)
pair.write("likes", 41)
print("primary:", pair.primary)
print("replica:", pair.replica, " <- a read here right now is stale")
pair.sync()
print("after sync, replica:", pair.replica)

**Questions to think about (answer in Part 8):** if a user pays £50 and immediately reloads their balance and the read goes to the replica, what do they see? Would a stale "likes" counter matter in the same way?

## 8. Reflection

1. **Partitioning.** Cluster B spread rows evenly but cluster C did not. Why? What real-world situations create "hot" keys (overloaded nodes)?
2. **What got faster?** The aggregate improved with four nodes but the single-trip lookup did not. Why? Under what conditions would the single node still be the better choice?
3. **Merging.** Why must the coordinator combine sums and counts rather than averages?
4. **Choosing the key.** In cluster C, what happened to lookups by `trip_id`, and why? What kind of query would C be good for?
5. **Failure.** Explain what happened in B and D when `node1` failed. What did replication cost you?
6. **Silent partial answers.** What is dangerous about `allow_partial=True`? How might a real system handle it?
7. **Stale reads.** Think about one example where a stale read is acceptable and one where it is not.
8. **Limits.** What are some of the things real distributed databases have to deal with that this simulation ignored?